In [9]:
# Data handling
import pandas as pd
import numpy as np

# Text preprocessing
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Sentiment analysis
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Feature extraction for ML models (if used)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Topic modeling
from sklearn.decomposition import LatentDirichletAllocation, NMF

# For display settings
import warnings
warnings.filterwarnings("ignore")
print('done')

done


In [11]:
# Mock Data Generation (Python):

import pandas as pd

import numpy as np




# Generate mock product review data

np.random.seed(42)

num_reviews = 1000

products = [f'Product_{chr(65+i)}' for i in range(5)] # Product_A, Product_B, ...

users = [f'User_{100+i}' for i in range(100)]



positive_phrases = [

    "love this product", "excellent quality", "works perfectly", "highly recommend",

    "great value", "very satisfied", "easy to use", "fantastic", "amazing features",

    "best purchase ever", "exceeded expectations", "five stars", "wonderful experience"

]

negative_phrases = [

    "terrible product", "poor quality", "does not work", "would not recommend",

    "waste of money", "very disappointed", "difficult to use", "awful", "missing features",

    "worst purchase", "broke easily", "one star", "bad experience", "customer service was bad"

]

neutral_phrases = [

    "it's okay", "average product", "works as expected", "nothing special", "decent for the price",

    "met expectations", "neither good nor bad", "could be better", "some pros and cons"

]



review_data = []

current_date = datetime(2023, 1, 1)



for i in range(num_reviews):

    rating = np.random.randint(1, 6)

    product_id = np.random.choice(products)

    user_id = np.random.choice(users)

    review_date = current_date + timedelta(days=np.random.randint(0, 365*2)) # Reviews over 2 years



    if rating >= 4:

        text = f"{np.random.choice(positive_phrases)}. {np.random.choice(positive_phrases)}."

        if np.random.rand() < 0.3: text += f" {np.random.choice(neutral_phrases)}."

    elif rating <= 2:

        text = f"{np.random.choice(negative_phrases)}. {np.random.choice(negative_phrases)}."

        if np.random.rand() < 0.3: text += f" {np.random.choice(neutral_phrases)}."

    else: # rating == 3

        text = f"{np.random.choice(neutral_phrases)}. "

        if np.random.rand() < 0.5: text += f"{np.random.choice(positive_phrases)}."

        else: text += f"{np.random.choice(negative_phrases)}."



    review_data.append({

        'ReviewID': f'REV{2000+i}',

        'ProductID': product_id,

        'UserID': user_id,

        'Rating': rating,

        'ReviewText': text,

        'ReviewDate': review_date.strftime('%Y-%m-%d')

    })



df_reviews = pd.DataFrame(review_data)



# Save to CSV

# df_reviews.to_csv('product_reviews_mock_data.csv', index=False)

# print("Mock product review data generated: product_reviews_mock_data.csv")

# print(df_reviews.head())

In [13]:
df=df_reviews.copy()

In [23]:
# Check shape (rows, columns)
print("Shape of dataset:", df.shape)

# Show column names
print("Columns:", df.columns.tolist())

# Check data types
df.info()

# Summary statistics (non-text fields)
df.describe()

# Count missing values
df.isnull().sum()


Shape of dataset: (1000, 6)
Columns: ['ReviewID', 'ProductID', 'UserID', 'Rating', 'ReviewText', 'ReviewDate']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ReviewID    1000 non-null   object
 1   ProductID   1000 non-null   object
 2   UserID      1000 non-null   object
 3   Rating      1000 non-null   int64 
 4   ReviewText  1000 non-null   object
 5   ReviewDate  1000 non-null   object
dtypes: int64(1), object(5)
memory usage: 47.0+ KB


ReviewID      0
ProductID     0
UserID        0
Rating        0
ReviewText    0
ReviewDate    0
dtype: int64

In [25]:
# Look at random samples of reviews
df[['ReviewText', 'Rating']].sample(10)

,ReviewText,Rating
222,amazing features. easy to use.,4
249,decent for the price. highly recommend.,3
64,exceeded expectations. highly recommend.,4
253,very satisfied. five stars.,5
624,love this product. very satisfied.,4
557,could be better. poor quality.,3
678,broke easily. customer service was bad.,1
502,difficult to use. waste of money.,1
526,neither good nor bad. highly recommend.,3
814,difficult to use. awful.,1


In [47]:
def clean_text(text):
    text = str(text).lower()                         # Lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)          # Remove punctuation, numbers, special chars
    return text

df['cleaned_review'] = df['ReviewText'].apply(clean_text)

In [49]:
df['cleaned_review'].head()

0         fantastic wonderful experience
1                     broke easily awful
2            met expectations five stars
3    very satisfied wonderful experience
4                worst purchase one star
Name: cleaned_review, dtype: object

In [51]:
df['ReviewText'].head()

0         fantastic. wonderful experience.
1                     broke easily. awful.
2            met expectations. five stars.
3    very satisfied. wonderful experience.
4                worst purchase. one star.
Name: ReviewText, dtype: object

In [55]:
print(df['ReviewText'].head())
print(type(df['ReviewText'][0]))

0         fantastic. wonderful experience.
1                     broke easily. awful.
2            met expectations. five stars.
3    very satisfied. wonderful experience.
4                worst purchase. one star.
Name: ReviewText, dtype: object
<class 'str'>


In [41]:
print(word_tokenize(df['cleaned_review'][0]))

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\dell/nltk_data'
    - 'C:\\Users\\dell\\anaconda3\\nltk_data'
    - 'C:\\Users\\dell\\anaconda3\\share\\nltk_data'
    - 'C:\\Users\\dell\\anaconda3\\lib\\nltk_data'
    - 'C:\\Users\\dell\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
    - 'C:/Users/dell/nltk_data'
**********************************************************************


In [43]:
import nltk
nltk.data.path.append('C:/Users/dell/nltk_data')

from nltk.tokenize import word_tokenize, sent_tokenize

text = "This is a test sentence."

print(sent_tokenize(text))           # Should work without error
print(word_tokenize(text))           # Should also work


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\dell/nltk_data'
    - 'C:\\Users\\dell\\anaconda3\\nltk_data'
    - 'C:\\Users\\dell\\anaconda3\\share\\nltk_data'
    - 'C:\\Users\\dell\\anaconda3\\lib\\nltk_data'
    - 'C:\\Users\\dell\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
    - 'C:/Users/dell/nltk_data'
    - 'C:/Users/dell/nltk_data'
**********************************************************************


In [ ]:
# Data Loading and Initial Exploration:
# Load the product review dataset.
# Examine the structure, review text, and ratings.
# Text Preprocessing:
# Clean the text data: convert to lowercase, remove punctuation, numbers, and special characters.
# Tokenize the reviews.
# Remove stop words.
# Perform stemming or lemmatization.
# Sentiment Analysis:
# Apply a pre-trained sentiment analyzer (e.g., VADER, TextBlob) to score each review.
# Alternatively, if labeled data were available (or can be partially labeled), train a simple classification model (e.g., Naive Bayes, Logistic Regression using TF-IDF or CountVectorizer features).
# Categorize reviews into positive, negative, and neutral sentiments.
# Exploratory Data Analysis of Sentiments:
# Analyze the distribution of sentiments.
# Create word clouds for positive and negative reviews to identify prominent terms.
# Examine sentiment trends by product (if applicable) or over time (if review dates are available).
# Topic Modeling (Optional but Recommended):
# Apply Latent Dirichlet Allocation (LDA) or Non-negative Matrix Factorization (NMF) to discover underlying topics in the reviews, especially within negative reviews to find common complaints.
# Insight Generation and Reporting:
# Summarize key findings: overall sentiment, common positive aspects, major pain points, and identified topics.
# Provide actionable recommendations based on the analysis.
# Deliverables:

# A Jupyter Notebook containing all code for text preprocessing, sentiment analysis, EDA, and topic modeling (if performed).
# A report summarizing the sentiment distribution, key themes, word clouds, and actionable insights for product and marketing teams.
# A presentation of the findings.